# One-dimensional cosine-potential Bloch bands

**Status:** Maintained calculated results for an idealized bare electron in a prescribed cosine potential. This is not a material-specific band structure or a DFT calculation.

**Prerequisite:** [One-dimensional cosine-potential Bloch bands](../../../../docs/lecturenotes/solidstate/bloch1d/cosinepotential/index.md)

This laboratory reuses the generic `pbc1d` boundary representation through package-owned Bloch Hamiltonians.


## Model

$$H=-\frac{\hbar^2}{2m_e}\frac{d^2}{dx^2}+V_0\cos\left(\frac{2\pi x}{a}\right),$$

with Bloch condition

$$\psi(x+a)=e^{ika}\psi(x).$$

At $V_0=0$, the represented operator reduces exactly to the `bloch1d/freeelectron` finite-difference Hamiltonian. A nonzero Fourier component couples free-electron modes separated by one reciprocal vector and opens the first zone-edge gap.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from projectkoios.physkit.solidstate.bloch1d.cell import BlochPrimitiveCell1D
from projectkoios.physkit.solidstate.bloch1d.cosinepotential.model import CosinePotentialBloch1D
from projectkoios.physkit.solidstate.bloch1d.freeelectron.model import FreeElectronBloch1D
from projectkoios.physkit.units import PhysicalUnit, ScalarQuantity, UnitSystem, VectorQuantity

cell = BlochPrimitiveCell1D(
    oriented_length=ScalarQuantity(10.0, UnitSystem.METAL.length_unit),
    unit_system=UnitSystem.METAL,
)
model = CosinePotentialBloch1D(
    cell=cell,
    amplitude=ScalarQuantity(0.5, UnitSystem.METAL.energy_unit),
)
reciprocal_unit = PhysicalUnit("1 / angstrom")


## Prescribed periodic potential

The amplitude is an illustrative model parameter, not a fitted material value.


In [ ]:
x = np.linspace(0.0, cell.length.magnitude, 400, endpoint=False)
potential = model.amplitude.magnitude * np.cos(2.0 * np.pi * x / cell.oriented_length.magnitude)
fig, ax = plt.subplots(figsize=(7.2, 3.8))
ax.plot(x, potential)
ax.set(xlabel="x (Å)", ylabel="V(x) (eV)", title="Prescribed cosine potential")
ax.grid(alpha=0.25)
fig.tight_layout()


## Bands and the free-electron reference

Solid curves are the represented cosine-potential eigenvalues. Dashed curves are exact folded free-electron branches. Their difference is the prescribed-potential effect plus finite-difference error.


In [ ]:
zone_edge = np.pi / cell.length.magnitude
k_values = np.linspace(-zone_edge, zone_edge, 161)
bands = model.solve_bands(
    bloch_wave_numbers=VectorQuantity(k_values, reciprocal_unit),
    point_count=64,
    band_count=5,
).energies.magnitude
free_model = FreeElectronBloch1D(cell=cell)
free_modes = tuple((n,) for n in range(-4, 5))
free_rows = []
for k in k_values:
    row = free_model.evaluate_modes(
        mode_indices=free_modes,
        bloch_wave_number=ScalarQuantity(float(k), reciprocal_unit),
    ).energies.magnitude
    free_rows.append(np.sort(row)[:5])
free_bands = np.asarray(free_rows)
fig, ax = plt.subplots(figsize=(7.4, 4.8))
ax.plot(k_values / zone_edge, free_bands, color="0.65", linestyle="--", linewidth=0.9)
ax.plot(k_values / zone_edge, bands, linewidth=1.2)
ax.set(xlabel=r"$k/(\pi/a)$", ylabel="Energy (eV)", title="Cosine-potential bands and folded free-electron reference")
ax.grid(alpha=0.25)
fig.tight_layout()


## First zone-edge gap

For a weak $V_0\cos(2\pi x/a)$ perturbation, the first gap approaches $|V_0|$. The calculation below reports represented values; it does not impose an empirical gap threshold.


In [ ]:
amplitudes = np.linspace(0.0, 1.0, 21)
gaps = []
for amplitude in amplitudes:
    candidate = CosinePotentialBloch1D(
        cell=cell,
        amplitude=ScalarQuantity(float(amplitude), UnitSystem.METAL.energy_unit),
    )
    edge = candidate.solve_bands(
        bloch_wave_numbers=VectorQuantity(np.array([zone_edge]), reciprocal_unit),
        point_count=80,
        band_count=2,
    ).energies.magnitude[0]
    gaps.append(edge[1] - edge[0])
fig, ax = plt.subplots(figsize=(6.8, 4.3))
ax.plot(amplitudes, gaps, marker="o", markersize=3, label="represented gap")
ax.plot(amplitudes, amplitudes, linestyle="--", color="black", label=r"weak-potential $|V_0|$")
ax.set(xlabel=r"$V_0$ (eV)", ylabel="First gap (eV)", title="Zone-edge gap versus cosine amplitude")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()


## Grid convergence

The zone-edge gap and individual energies have different discretization behavior. This plot checks the represented gap without claiming material convergence.


In [ ]:
point_counts = np.array([12, 16, 24, 32, 48, 64, 96])
represented_gaps = []
for point_count in point_counts:
    edge = model.solve_bands(
        bloch_wave_numbers=VectorQuantity(np.array([zone_edge]), reciprocal_unit),
        point_count=int(point_count),
        band_count=2,
    ).energies.magnitude[0]
    represented_gaps.append(edge[1] - edge[0])
fig, ax = plt.subplots(figsize=(6.8, 4.2))
ax.plot(point_counts, represented_gaps, marker="o")
ax.axhline(model.amplitude.magnitude, linestyle="--", color="black", label=r"weak-potential $|V_0|$")
ax.set(xlabel="Grid points per cell", ylabel="First gap (eV)", title="Represented gap under grid refinement")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()


## Interpretation and retained exploratory work

The package-backed calculation establishes the documented cosine-potential software model and its numerical behavior. It does not establish a semiconductor band gap.

The preserved exploratory notebooks retain additional density-of-states, finite-temperature chemical-potential, occupied/unoccupied DOS, thermal-window, and interactive presentation layers. Those layers remain pending separate contracts rather than being silently reproduced as notebook-local utilities.

## Exercises

1. Verify equality of the spectra at $k=-\pi/a$ and $k=\pi/a$.
2. Explain why changing the sign of $V_0$ leaves the complete spectrum unchanged.
3. Compare individual band-energy convergence with gap convergence.
4. Add a second Fourier harmonic through a new explicit potential contract.
5. Design a DOS estimator that records its energy grid, broadening or binning rule, and normalization.
